In [ ]:
# import sys
# !{sys.executable} -m pip install --user langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [2]:
from langchain_core.documents import Document

In [3]:
sample_doc = Document(
    page_content = "I Love My India",
    metadata = {"source": "http://www.google.com"}
)

In [4]:
sample_doc

Document(metadata={'source': 'http://www.google.com'}, page_content='I Love My India')

In [5]:
type(sample_doc)

langchain_core.documents.base.Document

In [6]:
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("data/python.txt", encoding="utf-8")

C:\Users\Sameer\AppData\Local\Temp\ipykernel_2652\3383052197.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [7]:
document = loader.load()

In [8]:
document

[Document(metadata={'source': 'data/python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and mo

In [9]:
# # PDF data

# # for normal data
# from langchain_community.document_loaders.pdf import PyPDFLoader
# pdf_loader = PyPDFLoader("data/research2.pdf")
# document = pdf_loader.load()
# document

# # for complex data
# from langchain_community.document_loaders.pdf import PyMuPDFLoader
# pdf_loader = PyMuPDFLoader("data/research2.pdf")
# document = pdf_loader.load()
# document

## Ingestion Pipeline

In [10]:
# Data => Documents
import os # for import our data from any path
from langchain_community.document_loaders.pdf import PyPDFLoader

In [11]:
def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print("total pdf", num_docs)
    print("total pages", len(all_docs))
    return all_docs

In [12]:
all_pdf_document = load_all_pdfs()

total pdf 1
total pages 21


In [13]:
type(all_pdf_document[20])

langchain_core.documents.base.Document

### Chunks

In [14]:
# chunks 
# %pip install langchain-text-splitters

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [16]:
chunks = split_docs(all_pdf_document)

In [17]:
len(chunks)

244

### Embedding

In [18]:
from sentence_transformers import SentenceTransformer

In [19]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):

        self.model_name=model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions", self.model.get_sentence_embedding_dimension())

    def generate_embedding(self, text):
        embedding = self.model.encode(text, show_progress_bar=True)
        print("embedding shape", embedding.shape)
        return embedding

In [20]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions 384


C:\Users\Sameer\AppData\Local\Temp\ipykernel_2652\3088930114.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions", self.model.get_sentence_embedding_dimension())


### Vector DB

In [21]:
import chromadb
import uuid 

In [22]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"discription": "vector store collection for pdf embedding in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")

        # store => ids, embedding, documment, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embedding_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_lenght"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embedding_list.append(embedding.tolist())

            self.collection.add(
                ids = ids,
                metadatas = all_metadata,
                documents = documents_content,
                embeddings = embedding_list
            )

        print("total document added in vector store", len(documents_content))
        print("docs in collection:", self.collection.count())

In [23]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 488


In [24]:
# data => document => chunks => embedding => store in vector store

texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embedding(texts)

vector_store.add_documents(chunks, embedding)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

embedding shape (244, 384)
total document added in vector store 244
docs in collection: 732


### Retrieval Pipeline

In [25]:
from sklearn.metrics.pairwise import cosine_similarity

In [26]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embedding([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k 
        )

        # cosine similarity
        retrieved_docs=[]

        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "ids": doc_id,
                        "documents": document,
                        "metadata": metadata,
                        "similarity_score": similarity_score,
                        "rank": i + 1 
                    })
                    
            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no document found")

        return retrieved_docs

In [27]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [28]:
rag_retriever.retrieve("What is encoder and decoder")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape (1, 384)
retrieved 0 documents


[]

### Integrate with LLMs

### Groq

In [ ]:
API_KEY_GROQ = "YOUR_GROQ_API_KEY"

In [36]:
# %pip install langchain-groq

In [55]:
from langchain_groq import ChatGroq

# Use a supported model name on Groq
llm = ChatGroq(
    groq_api_key=API_KEY_GROQ,
    model="openai/gpt-oss-120b",  # Alternative: "qwen-2.5-32b"
    temperature=0.1,
    max_tokens=1024
)

In [64]:
# generate our retrieval-augmented output 
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["documents"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query 
    prompt = f""" use given context to generate the answer for 
                query Context: {context}
                Query: {query}"""

    response = llm.invoke([prompt.format(context=context, query=query)]) # expecting a list as prompt
    return response.content

In [68]:
answer = generate_output("What is RAG", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:01<?, ?it/s]

embedding shape (1, 384)
retrieved 3 documents


In [69]:
print(answer) 

**RAG (Retrieval‑Augmented Generation)** is a paradigm for building language‑model‑based systems that combine two key steps:

1. **Retrieval** – the model first searches an external knowledge source (e.g., a document corpus, database, or the web) to pull in relevant passages or facts that are pertinent to the current query or task.  
2. **Generation** – a generative language model then uses both the original prompt **and** the retrieved information to produce its answer, summary, or continuation.

By grounding the generation step in up‑to‑date, factual material, RAG methods aim to improve the accuracy, relevance, and factual consistency of the output compared with a “pure” language model that relies only on its internal parameters.  

The survey you’re referencing reviews the evolution of RAG—from early “naïve” RAG approaches that simply concatenate retrieved texts with the prompt, to more sophisticated variants that integrate retrieval more tightly with the model’s architecture and tr